In [1]:
import pandas as pd
import numpy as np
from dl_client import DatalakeClient
import warnings
warnings.filterwarnings('ignore')

client = DatalakeClient()

In [2]:
def find_pop_variable(df, file_code, support_file):
    """
    Identifica la variabile di popolazione nei dati ADNI.
    
    Args:
        df (pd.DataFrame): DataFrame contenente i dati ADNI
        file_code (str): Codice del file per l'identificazione
        support_file (pd.DataFrame): DataFrame di supporto per salvare le informazioni
        
    Returns:
        str or None: Nome della variabile di popolazione identificata o None se non trovata
    """
    # Lista delle versioni ADNI da cercare
    adni_versions = ['ADNI1', 'ADNI2', 'ADNIGO', 'ADNI3', 'ADNI4']
    
    # Trova colonne che contengono valori di popolazione ADNI
    key_pop = []
    for key, value in df.items():
        values_list = value.tolist()
        if any(version in values_list for version in adni_versions):
            key_pop.append(key)
    
    # Determina la variabile di popolazione in base al numero di chiavi trovate
    if len(key_pop) == 1:
        pop = key_pop[0]
    elif len(key_pop) > 2:
        print(f"{file_code}\nPROBLEMA: più di 2 chiavi di popolazione trovate")
        print(key_pop)
        # Scegliamo il primo come default in questo caso
        pop = None
    elif len(key_pop) == 2:
        # Se le due colonne sono identiche, usa la prima
        if df[key_pop[0]].equals(df[key_pop[1]]):
            pop = key_pop[0]
        else:
            # Correggiamo la logica della variabile n
            n = 0
            pop = None
            for key in key_pop:
                # Controlla se tutti i valori per ogni RID sono unici
                if not df.groupby('RID')[key].nunique().eq(1).all():
                    n += 1
                    pop = key
            
            if n > 1:
                print(f"{file_code}\nPROBLEMA: 2 chiavi di popolazione diverse per soggetto")

            if n == 0:
                print(f"{file_code}\nPROBLEMA: 2 chiavi di popolazione diversa, ma con tutti valori uguali")
                pop = None
    else:
        print(f"{file_code}\nPROBLEMA: nessuna chiave di popolazione trovata")
        pop = None
    

    # Aggiorna il file di supporto se necessario
    support_file_new = support_file.copy(deep=True)
    if pop is not None:
        try:
            # Verifica se la variabile di popolazione è già presente nel file di supporto
            if pop not in support_file[support_file['file_code'] == file_code]['variable_code'].values:
                # Cerca l'indice corretto per inserire la nuova riga
                filtered_df = support_file[support_file['file_code'] == file_code]
                if not filtered_df.empty:
                    index = filtered_df.index[0] + 1
                    file_name = filtered_df['file_name'].iloc[0]
                    new_row = pd.Series({
                        'file_name': file_name, 
                        'file_code': file_code, 
                        'parameter': 'Cohort', 
                        'population': None, 
                        'variable_code': pop,
                        'type_variable': None,
                        'classes': None,
                        'range': None,
                        'valid_values': None,
                        'missing_values': None,
                        'missing_pop': None,
                        'del': False,
                    })
                    # Inserisci la nuova riga
                    support_file_new = pd.concat([
                        support_file.iloc[:index], 
                        pd.DataFrame([new_row]),
                        support_file.iloc[index:]
                    ]).reset_index(drop=True)
        except Exception as e:
            print(f"Errore nell'aggiornamento del file di supporto: {e}")
            support_file_new = support_file
    
    return pop, support_file_new

In [3]:
def select_variables(df, file_name, support_file, type='raw/'):
    metadata = client.get_metadata(
        object_name = type+file_name
    )

    file_code = metadata['metadata']['custom']['file_code']
    
    support_file = support_file[support_file['file_code']==file_code]
    lst_variable = [x for x in support_file['variable_code'].unique() if x in list(df.columns)] 
    df_new = df[lst_variable]

    return df_new

In [4]:
def check_missing_values(df, key, pop_key, stampa=False):  
    n_missing = int(df[key].isna().sum())
    n_tot = int(df.shape[0])
    n_valid = int(n_tot - n_missing)
    
    if pop_key is not None:
        pop = ['ADNI1', 'ADNIGO', 'ADNI2', 'ADNI3', 'ADNI4']
        pop_valid = df[df[key].isna() == False][pop_key].unique().tolist()
        pop_missing = [x for x in pop if x not in pop_valid]
    else:
        pop_valid = ['pop not found']
        pop_missing = ['pop not found']

    if stampa:
        print(f'Missing/total values:        {n_missing}/{n_tot}\nValid/totalvalues:          {n_valid}/{n_tot}')
        print('missing population:  ', pop_missing)
    
    return n_tot, n_valid, n_missing, pop_valid, pop_missing

In [5]:
def check_type_range_variables(df, key):
    n = df[key].first_valid_index()
    tipo = type(df[key][n])
    options = df[key].unique()
    if tipo is not str:
        intervallo = [float(df[key].max()), float(df[key].min())]
    else:
        intervallo = [None]
    
    if len(options) <= 10:
        classes = options
    elif tipo == str:
        classes = options[:5]
    else:
        classes = [None]

    return tipo, intervallo, classes

In [6]:
def get_varible_info(df, key, file_code, support_file, pop):
    n_tot, n_valid, n_missing, _, pop_missing = check_missing_values(df, key, pop)
    tipo, intervallo, classes = check_type_range_variables(df, key)

    index = support_file.index[(support_file['file_code'] == file_code) & (support_file['variable_code'] == key)][0]

    support_file['type_variable'][index] = tipo
    support_file['classes'][index] = ', '.join(map(str, classes))
    support_file['range'][index] = ', '.join(map(str, intervallo))
    support_file['valid_values'][index] = int(n_valid)
    support_file['missing_values'][index] = int(n_missing)
    support_file['missing_pop'][index] = ', '.join(map(str, pop_missing))
    
    if n_valid/n_tot <= 0.65:
        support_file['del'][index] = True
    else:
        support_file['del'][index] = False

    return support_file

In [7]:
search = client.query_files(
    query={'custom.level' : 'raw'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(zip_files.keys())

support_file = pd.read_excel('../ADNI_variables_statistics.xlsx')

dict_keys(['ADNIMERGE_06Jun2025.csv', 'ADSP_PHC_BIOMARKER_06Jun2025.csv', 'BLCHANGE_06Jun2025.csv', 'DXSUM_06Jun2025.csv', 'MMSE_06Jun2025.csv', 'NEUROPATH_06Jun2025.csv', 'PTDEMOG_06Jun2025.csv'])


> codice per un singolo file

In [8]:
file_name = 'ADSP_PHC_BIOMARKER_06Jun2025.csv'
df = zip_files[file_name]
file_code = file_name[:-14]
pop, support_file = find_pop_variable(df, file_code, support_file)
for key in df.keys():
    if key in support_file[support_file['file_code'] == file_code]['variable_code'].values:
        get_varible_info(df, key, file_code, support_file, pop)

support_file.to_excel('ADNI_variables_statistics.xlsx', index=False)
support_file.to_csv('ADNI_variables_statistics.csv', index=False)

> codice per iterare sopra tutti i file estratti dalla query

In [ ]:
for file_name in zip_files.keys():
    df = zip_files[file_name]
    file_code = file_name[:-14]
    pop, support_file = find_pop_variable(df, file_code, support_file)
    for key in df.keys():
        if key in support_file[support_file['file_code'] == file_code]['variable_code'].values:
            get_varible_info(df, key, file_code, support_file, pop)

support_file.to_excel('ADNI_variables_statistics.xlsx', index=False)
support_file.to_csv('ADNI_variables_statistics.csv', index=False)

## Test su singolo file

In [ ]:
file_name = 'ADNIMERGE_06Jun2025.csv'
df = zip_files[file_name]

support_file = pd.read_excel('ADNI_variables_statistics.xlsx')
df_new = select_variables(df, file_name, support_file)

In [ ]:
file_code = 'ADNIMERGE'
for key in df_new.keys():
    get_varible_info(df_new, key, file_code, support_file)

In [ ]:
support_file

In [ ]:
support_file.to_excel('ADNI_variables_statistics.xlsx', index=False)

In [ ]:
df_new['DX'].unique()

In [ ]:
dd_boh = df.apply(lambda x: x.ORIGPROT == x.COLPROT, axis=1)

In [ ]:
dd_boh[dd_boh == False].index

In [ ]:
df_new[df_new['PTRACCAT']=='Black']

In [ ]:
df_new[df_new['AV45'].isna() == False]